# Travail en équipe 60600 (A2026) : partie A

**Partie A** : prédire le premier jeu installé le mois suivant (Y = 1, 2, 3 ou 4), critère = accuracy.
Point de référence de l'énoncé : forêt `ranger` par défaut, erreur OOB de 28,24 % (accuracy d'environ 71,8 %).

Le notebook s'exécute de haut en bas (*Restart & Run All*). Durée indicative : 20 à 30 min dans Colab,
surtout pour les étapes Optuna (on peut baisser `N_ESSAIS_A` et `N_ESSAIS_ORD` pour un essai rapide).

In [ ]:
# ============================================================================
# PACKAGES REQUIS (Python >= 3.9)
#   pandas, numpy, matplotlib, scikit-learn (>= 1.4), optuna (>= 3.0)
#   Installation au besoin : pip install pandas numpy matplotlib scikit-learn optuna
# ============================================================================
import warnings
from itertools import combinations, permutations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor)
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)   # n'affiche pas chaque essai
print("scikit-learn", sklearn.__version__, "| optuna", optuna.__version__)

DOSSIER = "./"   # dossier contenant train.csv et test.csv
SEED = 60600     # graine unique : tous les résultats sont reproductibles
CATEG = ["country", "difflevel"]   # variables traitées comme catégorielles, comme dans le cours

---
# Partie A : classification à 4 classes

## A1. Données

In [ ]:
# Charger les données
train = pd.read_csv(DOSSIER + "train.csv")
test = pd.read_csv(DOSSIER + "test.csv")

# Afficher les dimensions
print("Train :", train.shape)
print("Test :", test.shape)

# Afficher les premières lignes
display(train.head())
display(test.head())

In [ ]:
# Distribution de la variable cible Y
display(pd.DataFrame({"n": train["y"].value_counts().sort_index(),
                      "proportion": train["y"].value_counts(normalize=True).sort_index().round(3)}))

In [ ]:
# Statistiques descriptives et valeurs manquantes
display(train.describe().T.round(3))
print("Valeurs manquantes - train :", int(train.isna().sum().sum()), "| test :", int(test.isna().sum().sum()))

Les 4 classes ont des fréquences presque égales (environ 25 % chacune) : l'accuracy est donc un critère
adéquat et le modèle naïf ne fait qu'environ 25 %. Par contre, en **OVA** chaque problème binaire oppose
une classe à trois autres (environ 25 % contre 75 %), le déséquilibre que Galar et al. (2011) identifient
comme la principale faiblesse d'OVA. En **OVO**, chaque paire reste équilibrée (environ 50/50).

Comme dans le cours, `country` (6 régions) et `difflevel` (3 niveaux) sont traitées comme catégorielles par le boosting.

In [ ]:
# Préparer les variables explicatives
def categ(df, refs):
    """Variables explicatives avec country et difflevel en type 'category'.
    Les modalités sont fixées à partir de refs (train + test) pour être identiques partout."""
    X = df.drop(columns=[c for c in ("y", "id") if c in df]).copy()
    for c in CATEG:
        X[c] = pd.Categorical(X[c], categories=sorted(set().union(*(r[c] for r in refs))))
    return X

def num(X):
    """Version entièrement numérique (codes entiers) pour la forêt aléatoire de scikit-learn."""
    return X.astype({c: int for c in CATEG})

X, y = categ(train, [train, test]), train["y"]
X_test = categ(test, [train, test])
CLASSES = sorted(y.unique())
print("Variables explicatives :", list(X.columns))
print("Classes :", CLASSES)

## A2. Protocole d'évaluation
On combine les deux approches vues en classe (division de l'échantillon + validation croisée) :

1. **1 000 observations mises de côté** (« test interne ») : elles servent **une seule fois**, à la fin,
   pour confirmer la performance des modèles retenus ;
2. sur les **4 000 autres**, une **validation croisée stratifiée à 5 plis** sert à comparer les méthodes et à
   optimiser les hyperparamètres. Toutes les méthodes utilisent **les mêmes plis** (comparaison appariée).

On rapporte la moyenne et l'erreur-type (ET) de l'accuracy entre les plis.
Avec 1 000 observations, l'erreur-type d'une accuracy d'environ 0,8 est d'environ √(0,8×0,2/1000) ≈ 1,3 point :
des écarts de quelques dixièmes de point entre deux méthodes ne sont pas significatifs.

In [ ]:
# Diviser : 4 000 obs. de développement / 1 000 obs. de test interne (stratifié selon Y)
X_dev, X_hold, y_dev, y_hold = train_test_split(X, y, test_size=1000, stratify=y, random_state=SEED)
print("Développement :", X_dev.shape, "| Test interne :", X_hold.shape)

# Créer les 5 plis de validation croisée (les mêmes pour toutes les méthodes)
PLIS = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_dev, y_dev))
print("Taille des plis de validation :", [len(va) for tr, va in PLIS])

In [ ]:
# Fonctions d'évaluation par validation croisée
def val_croisee(methode, params=None):
    """Validation croisée à 5 plis sur les 4 000 obs. de développement.
    methode(X_tr, y_tr, X_va, params) retourne un dict {nom: prédictions}.
    Retourne un dict {nom: liste des 5 accuracy}."""
    acc = {}
    for tr, va in PLIS:
        preds = methode(X_dev.iloc[tr], y_dev.iloc[tr], X_dev.iloc[va], params)
        for nom, p in preds.items():
            acc.setdefault(nom, []).append(accuracy_score(y_dev.iloc[va], p))
    return acc

def resume(acc):
    """Tableau : accuracy moyenne en VC et son erreur-type."""
    return pd.DataFrame({"acc_VC": {n: np.mean(v) for n, v in acc.items()},
                         "ET": {n: np.std(v, ddof=1) / np.sqrt(len(v)) for n, v in acc.items()}})

resultats = {}   # accumule les accuracy par pli de toutes les méthodes

## A3. Points de référence : modèle naïf et forêt aléatoire directe
Pour la forêt aléatoire, `mtry` (`max_features`) est généralement **l'hyperparamètre le plus important**,
alors que le nombre d'arbres n'a pas à être optimisé : il suffit d'en avoir assez (notes de cours). On le choisit
avec l'**erreur OOB**, sans avoir à diviser l'échantillon. 500 arbres et des feuilles de taille 1 correspondent
au `ranger` par défaut de l'énoncé.

In [ ]:
# Modèle naïf : prédit toujours la classe la plus fréquente de l'échantillon d'entrainement
def naif(X_tr, y_tr, X_va, params):
    return {"Naïf (classe majoritaire)": np.full(len(X_va), y_tr.mode()[0])}

resultats.update(val_croisee(naif))
display(resume(resultats).round(4))

In [ ]:
# Forêt aléatoire : choisir mtry avec l'accuracy OOB (sur les 4 000 obs. de développement)
oob = {}
for m in [2, 3, 4, 6, 8, 10, 12, 16]:
    rf_m = RandomForestClassifier(n_estimators=500, max_features=m, oob_score=True,
                                  n_jobs=-1, random_state=SEED).fit(num(X_dev), y_dev)
    oob[m] = rf_m.oob_score_

display(pd.DataFrame({"accuracy OOB": oob}).rename_axis("mtry").round(4))
MTRY = max(oob, key=oob.get)
print("mtry retenu :", MTRY)

In [ ]:
# Forêt aléatoire directe : validation croisée avec mtry=4 (défaut) et mtry choisi par OOB
def foret(X_tr, y_tr, X_va, params):
    sortie = {}
    for m, nom in [(4, "RF direct (mtry=4, défaut)"), (MTRY, f"RF direct (mtry={MTRY}, OOB)")]:
        rf_m = RandomForestClassifier(n_estimators=500, max_features=m, n_jobs=-1, random_state=SEED)
        sortie[nom] = rf_m.fit(num(X_tr), y_tr).predict(num(X_va))
    return sortie

resultats.update(val_croisee(foret))
display(resume(resultats).round(4))

## A4. Boosting : approche directe ou décomposition binaire (Galar et al., 2011)
On compare trois stratégies avec **le même classifieur de base** (boosting d'arbres) et **les mêmes
hyperparamètres de départ** `P0`, comme dans l'étude de Galar et al. Le classifieur de base est
`HistGradientBoosting` de scikit-learn, qui suit le même principe que LightGBM utilisé en classe
(histogrammes, arbres construits feuille par feuille) et traite directement les variables catégorielles.

* **Direct** : un seul modèle de boosting multi-classes (perte multinomiale) ;
* **OVA** (*one-vs-all*) : 4 modèles « classe k contre les autres ». On prédit la classe dont le modèle
  donne la plus grande confiance (agrégation **MAX**) ;
* **OVO** (*one-vs-one*) : 6 modèles, un par paire de classes (a, b), chacun entrainé seulement sur les
  observations des classes a et b. Deux agrégations de l'article :
  **VOTE** (chaque modèle vote pour une classe) et **WV** (vote pondéré : on additionne les confiances
  r_ab et 1 − r_ab), que l'article classe parmi les agrégations les plus robustes.

Dans le boosting, le nombre d'itérations doit être optimisé (risque de sur-ajustement), le taux
d'apprentissage doit simplement être petit, et la taille des arbres a souvent un impact important (notes de cours).
Ici, le nombre d'itérations est fixé par **arrêt précoce sur 10 % de l'échantillon d'entrainement** : c'est
l'idée de l'exemple LightGBM du cours (choisir les itérations sur une partie validation), mais faite à
l'intérieur de chaque entrainement. Le pli de validation n'est jamais touché, donc **pas de fuite d'information**.

In [ ]:
# Classifieur de base : boosting d'arbres avec arrêt précoce
def booster(params, regression=False):
    """Boosting d'arbres (HistGradientBoosting, même famille d'algorithme que LightGBM).
    - country et difflevel sont traitées directement comme catégorielles (from_dtype) ;
    - le nombre d'itérations est choisi par arrêt précoce sur 10 % de l'échantillon
      d'ENTRAINEMENT : le pli de validation n'est jamais utilisé pour l'ajuster."""
    classe = HistGradientBoostingRegressor if regression else HistGradientBoostingClassifier
    return classe(**params, max_iter=2000, early_stopping=True, validation_fraction=0.1,
                  n_iter_no_change=20, categorical_features="from_dtype", random_state=SEED)

# Hyperparamètres de départ, identiques pour les 3 stratégies
P0 = dict(learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=20,
          l2_regularization=0.0, max_features=1.0)
print("P0 :", P0)

In [ ]:
# Stratégie 1 : boosting direct (un seul modèle à 4 classes)
def direct(X_tr, y_tr, X_va, params):
    return {"Boosting direct": booster(params).fit(X_tr, y_tr).predict(X_va)}

resultats.update(val_croisee(direct, P0))
display(resume(resultats).round(4))

In [ ]:
# Stratégie 2 : OVA, 4 modèles « classe k contre les 3 autres », agrégation MAX
def ova(X_tr, y_tr, X_va, params):
    # Colonne k = confiance que l'observation appartient à la classe k
    conf = np.column_stack([booster(params).fit(X_tr, y_tr == k).predict_proba(X_va)[:, 1]
                            for k in CLASSES])
    return {"Boosting OVA (MAX)": np.array(CLASSES)[conf.argmax(axis=1)]}

resultats.update(val_croisee(ova, P0))
display(resume(resultats).round(4))

In [ ]:
# Stratégie 3 : OVO, un modèle binaire par paire de classes (6 modèles)
def ajuster_ovo(X_tr, y_tr, params):
    # Le modèle (a, b) est entrainé seulement sur les obs. des classes a et b
    modeles = {}
    for a, b in combinations(CLASSES, 2):
        garder = y_tr.isin([a, b])
        modeles[(a, b)] = booster(params).fit(X_tr[garder], y_tr[garder] == a)
    return modeles

def scores_ovo(modeles, X_va, agregation="WV"):
    """Agrégation des 6 modèles (Galar et al., 2011) : VOTE ou WV (vote pondéré). Score de chaque classe."""
    score = pd.DataFrame(0.0, index=range(len(X_va)), columns=CLASSES)
    for (a, b), mod in modeles.items():
        r_ab = mod.predict_proba(X_va)[:, 1]          # confiance « classe a plutôt que b »
        if agregation == "VOTE":
            r_ab = (r_ab > 0.5).astype(float)         # vote 0/1
        score[a] += r_ab
        score[b] += 1 - r_ab
    return score

def predire_ovo(modeles, X_va, agregation="WV"):
    return scores_ovo(modeles, X_va, agregation).idxmax(axis=1).to_numpy()   # classe au plus grand score

def ovo(X_tr, y_tr, X_va, params):
    modeles = ajuster_ovo(X_tr, y_tr, params)
    return {"Boosting OVO (VOTE)": predire_ovo(modeles, X_va, "VOTE"),
            "Boosting OVO (WV)": predire_ovo(modeles, X_va, "WV")}

In [ ]:
# Validation croisée de l'OVO (les deux agrégations utilisent les mêmes 6 modèles)
resultats.update(val_croisee(ovo, P0))
display(resume(resultats).round(4))

## A4 bis. Chercher la structure : variables dérivées et modèle paramétrique
Des équipes atteignent environ 88 % sur le tableau public, ce qui veut dire qu'il reste du signal que le
boosting sur les 16 variables brutes ne capte pas. Les données sont simulées : si Y dépend de **ratios** ou
d'**interactions**, les arbres doivent les approximer par une série de coupures, ce qui demande beaucoup
d'observations. Deux pistes inspirées du cours :

1. **Variables dérivées** (ratios « par unité ») ajoutées aux 16 variables, par exemple le montant moyen par achat.
   Dans l'exemple des jeux du cours, l'interaction la plus forte (H-index) est `numpurchases` × `totpurchases`.
2. **Régression logistique multinomiale avec les carrés et les interactions d'ordre 2**, l'équivalent pour Y
   catégoriel du benchmark « Rég lin (carrés et inter d'ordre 2) » du cours, qui fait un très grand saut par rapport
   à la régression simple. Avec plusieurs centaines de termes, on ajoute une pénalité ridge (L2) dont la force
   (paramètre C) est choisie par validation croisée. Les variables sont **standardisées avec les moyennes et
   écarts-types de l'échantillon d'entrainement seulement** (le pipeline est réajusté dans chaque pli), comme le
   recommande le cours pour ne pas contaminer les données de validation.

In [ ]:
# Créer les variables dérivées (ratios « par unité »)
def derivees(X):
    """Ajoute 7 ratios aux 16 variables (dénominateur ramené à au moins 1)."""
    X = X.copy()
    ratio = lambda a, b: X[a] / X[b].clip(lower=1)
    X["achat_moyen"] = ratio("totpurchases", "numpurchases")     # montant moyen par achat
    X["achats_session"] = ratio("numpurchases", "numsessions")   # achats par session
    X["sessions_jour"] = ratio("numsessions", "numdays")         # sessions par jour joué
    X["temps_session"] = ratio("totplaytime", "numsessions")     # minutes par session
    X["temps_jour"] = ratio("totplaytime", "numdays")            # minutes par jour joué
    X["vies_element"] = ratio("numlives", "numelements")         # vies utilisées par niveau réussi
    X["score_element"] = ratio("totscore", "numelements")        # score par niveau réussi
    return X

def avec_derivees(methode):
    """Même méthode, mais appliquée aux variables originales + dérivées."""
    def f(X_tr, y_tr, X_va, params):
        preds = methode(derivees(X_tr), y_tr, derivees(X_va), params)
        return {nom + " + dérivées": p for nom, p in preds.items()}
    return f

# Afficher les nouvelles variables
display(derivees(X).iloc[:, -7:].head().round(3))

In [ ]:
# Boosting (3 stratégies) avec les variables dérivées, mêmes hyperparamètres P0
for methode in (direct, ova, ovo):
    resultats.update(val_croisee(avec_derivees(methode), P0))
display(resume(resultats).round(4))

In [ ]:
# Régression logistique multinomiale avec carrés et interactions d'ordre 2
def nom_logit(params):
    return f"Logistique carrés+inter. (C={params['C']})" + (" + dérivées" if params["derivees"] else "")

def modele_logit(C):
    """Indicatrices pour country et difflevel, standardisation, puis tous les carrés et
    interactions d'ordre 2, standardisés à nouveau, et pénalité ridge de force 1/C."""
    return make_pipeline(
        ColumnTransformer([("cat", OneHotEncoder(drop="first"), CATEG)], remainder=StandardScaler()),
        PolynomialFeatures(degree=2, include_bias=False), StandardScaler(),
        LogisticRegression(C=C, max_iter=5000))

def logistique(X_tr, y_tr, X_va, params):
    transfo = derivees if params["derivees"] else (lambda Z: Z)
    return {nom_logit(params): modele_logit(params["C"]).fit(transfo(X_tr), y_tr).predict(transfo(X_va))}

In [ ]:
# Recherche en grille sur C, avec et sans variables dérivées
P_LOGIT = [{"C": C, "derivees": d} for d in (False, True) for C in (0.001, 0.01, 0.1, 1.0)]
for p in P_LOGIT:
    resultats.update(val_croisee(logistique, p))
display(resume({nom_logit(p): resultats[nom_logit(p)] for p in P_LOGIT}).round(4))

In [ ]:
# Résumé de toutes les méthodes comparées jusqu'ici (triées par accuracy en VC)
display(resume(resultats).sort_values("acc_VC", ascending=False).round(4))

**Importance des variables par permutation** (méthode du cours) pour le boosting direct avec variables dérivées :
le modèle est entrainé sur les 4 plis d'entrainement du premier découpage, puis on mesure la perte d'accuracy
sur le pli de validation quand on permute aléatoirement chaque variable. Les variables du haut indiquent où
chercher d'autres transformations utiles.

In [ ]:
# Importance des variables par permutation
tr, va = PLIS[0]
mod_vimp = booster(P0).fit(derivees(X_dev.iloc[tr]), y_dev.iloc[tr])
vimp = permutation_importance(mod_vimp, derivees(X_dev.iloc[va]), y_dev.iloc[va], scoring="accuracy",
                              n_repeats=5, random_state=SEED)
vimp = pd.Series(vimp.importances_mean, index=derivees(X_dev).columns).sort_values()
vimp.plot.barh(figsize=(6, 6), title="Importance par permutation (baisse d'accuracy)")
plt.tight_layout(); plt.show()

## A4 ter. Les classes sont-elles ordonnées ? Sélection de variables
Dans une exécution précédente, la matrice de confusion du stacking montrait que les erreurs se font presque
uniquement entre « voisins » d'une chaîne de classes (2 – 3 – 4 – 1), jamais entre classes éloignées.
C'est ce qu'on observe quand Y est obtenue en **découpant une seule échelle continue cachée** en 4 intervalles.
On le vérifie ici avec des prédictions **hors-pli** sur les 4 000 obs. de développement (le test interne n'est
pas utilisé), et l'ordre est choisi automatiquement : la chaîne qui maximise les confusions entre classes voisines.

La même boucle calcule l'**importance par permutation en validation croisée** (moyenne sur les 5 plis). Elle sert
à retirer les variables inutiles (sélection de variables, notes « suite ») : on garde les variables dont la
permutation fait perdre au moins 0,5 point d'accuracy.

In [ ]:
# Importance par permutation d'une méthode (utilisée ici et pour le modèle final en A6)
def vimp_permutation(methode, cle, params, X_tr, y_tr, X_va, y_va, R=5):
    """Le modèle est estimé UNE fois sur (X_tr, y_tr) ; on prédit en même temps X_va et ses copies
    où une variable est permutée (R fois par variable). Retourne (prédictions sur X_va, importances)."""
    rng = np.random.default_rng(SEED)
    copies = [X_va]
    for v in X_va.columns:
        for r in range(R):
            Xp = X_va.copy()
            Xp[v] = Xp[v].iloc[rng.permutation(len(Xp))].set_axis(Xp.index)
            copies.append(Xp)
    pred = methode(X_tr, y_tr, pd.concat(copies, ignore_index=True), params)[cle]
    n = len(X_va)
    acc = [accuracy_score(y_va, pred[k * n:(k + 1) * n]) for k in range(len(copies))]
    imp = pd.Series({v: acc[0] - np.mean(acc[1 + i * R: 1 + (i + 1) * R]) for i, v in enumerate(X_va.columns)})
    return pred[:n], imp

# Prédictions hors-pli et importances de l'OVO (WV) dans chacun des 5 plis
pred_oof = np.zeros(len(y_dev), dtype=int)
vimp_plis = []
for tr, va in PLIS:
    pred_oof[va], imp = vimp_permutation(ovo, "Boosting OVO (WV)", P0, X_dev.iloc[tr], y_dev.iloc[tr],
                                         X_dev.iloc[va], y_dev.iloc[va], R=3)
    vimp_plis.append(imp)
print("Accuracy hors-pli de l'OVO (WV) :", round(accuracy_score(y_dev, pred_oof), 4))

In [ ]:
# Matrice de confusion hors-pli (lignes = vraie classe, colonnes = prédite)
mc_oof = confusion_matrix(y_dev, pred_oof, labels=CLASSES)
display(pd.DataFrame(mc_oof, index=[f"vrai {k}" for k in CLASSES], columns=[f"prédit {k}" for k in CLASSES]))

# Ordre des classes : la chaîne qui maximise les confusions entre classes voisines
conf = mc_oof + mc_oof.T                      # confusions dans les deux sens
pos = {k: i for i, k in enumerate(CLASSES)}
voisins = lambda ordre: sum(conf[pos[a], pos[b]] for a, b in zip(ordre, ordre[1:]))
ORDRE = list(max(permutations(CLASSES), key=voisins))
RANG = {k: r for r, k in enumerate(ORDRE, start=1)}           # classe -> rang sur l'échelle cachée
erreurs = mc_oof.sum() - np.trace(mc_oof)
print("Ordre retenu :", " < ".join(str(k) for k in ORDRE))
print(f"Part des erreurs entre classes voisines : {voisins(ORDRE) / erreurs:.1%}")

In [ ]:
# Importance moyenne sur les 5 plis et sélection des variables utiles (au moins 0,5 point d'accuracy)
vimp_vc = pd.DataFrame(vimp_plis)
tab_vimp = pd.DataFrame({"importance": vimp_vc.mean(),
                         "ET": vimp_vc.std(ddof=1) / np.sqrt(len(vimp_vc))}).sort_values("importance", ascending=False)
display(tab_vimp.round(4))
VARS_UTILES = list(tab_vimp.index[tab_vimp["importance"] >= 0.005])
print("Variables retenues :", VARS_UTILES)
print("Variables retirées :", [v for v in X.columns if v not in VARS_UTILES])

def avec_selection(methode):
    """Même méthode, mais avec seulement les variables retenues."""
    def f(X_tr, y_tr, X_va, params):
        preds = methode(X_tr[VARS_UTILES], y_tr, X_va[VARS_UTILES], params)
        return {nom + " + sélection": p for nom, p in preds.items()}
    return f

### Deux approches qui exploitent l'ordre des classes
* **Décomposition ordinale** (Frank et Hall, 2001) : 3 modèles binaires « rang > k » (k = 1, 2, 3), chacun estimé
  avec **toutes** les observations (contre environ la moitié pour un modèle OVO). On en déduit
  P(rang = r) = P(rang > r − 1) − P(rang > r). C'est une décomposition binaire, comme OVO et OVA dans Galar et al.,
  mais adaptée à des classes ordonnées.
* **Régression sur le rang** : un boosting de **régression** (Y continu) estime directement l'échelle cachée à partir
  du rang (1 à 4). La prédiction est ensuite découpée par 3 seuils, choisis pour maximiser l'accuracy sur des
  prédictions hors-pli internes de l'échantillon d'entrainement (le pli de validation n'est jamais utilisé).

In [ ]:
# Décomposition ordinale (Frank et Hall, 2001) : 3 modèles binaires « rang > k »
def probas_ordinal(X_tr, y_tr, X_va, params):
    """Probabilités des 4 classes (colonnes dans l'ordre de CLASSES)."""
    rang_tr = y_tr.map(RANG)
    p_sup = np.column_stack([booster(params).fit(X_tr, rang_tr > k).predict_proba(X_va)[:, 1]
                             for k in range(1, len(ORDRE))])           # P(rang > k), k = 1, 2, 3
    p_sup = np.minimum.accumulate(p_sup, axis=1)                       # assure P(>1) >= P(>2) >= P(>3)
    un, zero = np.ones((len(X_va), 1)), np.zeros((len(X_va), 1))
    p_rang = -np.diff(np.hstack([un, p_sup, zero]), axis=1)            # P(rang = r) = P(> r-1) - P(> r)
    return p_rang[:, [RANG[k] - 1 for k in CLASSES]]

def ordinal(X_tr, y_tr, X_va, params):
    P = probas_ordinal(X_tr, y_tr, X_va, params)
    return {"Boosting ordinal (Frank-Hall)": np.array(CLASSES)[P.argmax(axis=1)]}

In [ ]:
# Régression sur le rang : boosting de régression, puis découpage par 3 seuils
def seuils_optimaux(score, rang):
    """Seuils t1 < t2 < t3 qui maximisent l'accuracy du découpage (recherche coordonnée sur une grille)."""
    t = np.array([1.5, 2.5, 3.5])
    grille = np.quantile(score, np.linspace(0.01, 0.99, 197))
    for _ in range(3):
        for j in range(3):
            bas = t[j - 1] if j > 0 else -np.inf
            haut = t[j + 1] if j < 2 else np.inf
            cand = grille[(grille > bas) & (grille < haut)]
            if len(cand):
                acc = [np.mean(np.digitize(score, np.r_[t[:j], c, t[j + 1:]]) + 1 == rang) for c in cand]
                t[j] = cand[int(np.argmax(acc))]
    return t

def score_rang(X_tr, y_tr, X_va, params):
    """Score prédit pour X_va par un boosting de régression sur le rang (1 à 4)."""
    return booster(params, regression=True).fit(X_tr, y_tr.map(RANG)).predict(X_va)

def regression_rang(X_tr, y_tr, X_va, params):
    # Seuils choisis sur des prédictions hors-pli internes (5 plis) de l'échantillon d'entrainement
    rang_tr = y_tr.map(RANG).to_numpy()
    score_oof = np.zeros(len(X_tr))
    for tr, va in KFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr):
        score_oof[va] = score_rang(X_tr.iloc[tr], y_tr.iloc[tr], X_tr.iloc[va], params)
    t = seuils_optimaux(score_oof, rang_tr)
    rang_pred = np.digitize(score_rang(X_tr, y_tr, X_va, params), t) + 1
    return {"Régression sur le rang": np.array(ORDRE)[rang_pred - 1]}

In [ ]:
# Validation croisée des approches ordinales et effet de la sélection de variables (hyperparamètres P0)
for methode in (ordinal, regression_rang):
    resultats.update(val_croisee(methode, P0))
for methode in (ovo, ordinal, regression_rang):
    resultats.update(val_croisee(avec_selection(methode), P0))
display(resume(resultats).sort_values("acc_VC", ascending=False).head(10).round(4))

## A5. Optimisation des hyperparamètres (lien avec la partie B)
On optimise les deux stratégies de boosting finalistes, **direct** et **OVO (WV)**, chacune avec ou sans
variables dérivées selon ce qui a donné la meilleure VC en A4 bis. Les deux ont la même méthode et le même budget :
**optimisation bayésienne** (échantillonneur TPE d'Optuna, `N_ESSAIS_A` essais chacun). Le critère est
l'accuracy moyenne de la validation croisée à 5 plis ci-dessus. Pour OVO, un même jeu d'hyperparamètres est
utilisé pour les 6 modèles binaires (chacun garde son propre nombre d'itérations par arrêt précoce).

Comme les hyperparamètres sont choisis en maximisant cette accuracy de VC, celle du meilleur essai est
légèrement **optimiste**. C'est pourquoi on confirme ensuite sur le test interne (A6).

Les deux approches ordinales sont optimisées de la même façon, avec un **espace élargi** (`max_leaf_nodes` à
partir de 2, car l'optimisation de l'OVO avait choisi la borne inférieure 4) et `N_ESSAIS_ORD` essais, chacune avec
ou sans sélection de variables selon la meilleure VC. Les études du boosting direct et de l'OVO ne changent pas :
le stacking soumis précédemment (84,2 % sur Kaggle) reste ainsi exactement reproductible.

In [ ]:
# Espace de recherche des hyperparamètres du boosting
def espace(trial):
    return dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),    # rétrécissement
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 4, 64, log=True),        # taille des arbres
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 100, log=True),   # taille min. d'une feuille
        l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
        max_features=trial.suggest_float("max_features", 0.3, 1.0))                 # analogue de mtry

def espace_large(trial):
    """Même espace, mais les arbres peuvent descendre à 2 feuilles (souches)."""
    return dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 2, 64, log=True),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 100, log=True),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-4, 10, log=True),
        max_features=trial.suggest_float("max_features", 0.3, 1.0))

# Pour chaque stratégie, garder la version (avec ou sans variables dérivées) qui a la meilleure VC
acc_moy = lambda nom: np.mean(resultats[nom])

def version_choisie(nom, methode, suffixe=" + dérivées", variante=avec_derivees):
    if acc_moy(nom + suffixe) > acc_moy(nom):
        return nom + suffixe, variante(methode)
    return nom, methode

FIN_DIRECT = version_choisie("Boosting direct", direct)
FIN_OVO = version_choisie("Boosting OVO (WV)", ovo)
print("Finalistes :", FIN_DIRECT[0], "|", FIN_OVO[0])

In [ ]:
# Fonction d'optimisation bayésienne (TPE) d'une méthode
N_ESSAIS_A = 25
OPTIMISES = {}   # nom -> (méthode, meilleurs hyperparamètres)

def optimiser(nom, methode, espace_fn=espace, n_essais=N_ESSAIS_A):
    def objectif(trial):
        acc = [float(a) for a in val_croisee(methode, espace_fn(trial))[nom]]
        trial.set_user_attr("acc_plis", acc)       # garde l'accuracy de chaque pli
        return float(np.mean(acc))
    etude = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    etude.optimize(objectif, n_trials=n_essais)
    resultats[nom + " optimisé"] = etude.best_trial.user_attrs["acc_plis"]
    OPTIMISES[nom + " optimisé"] = (methode, etude.best_params)
    print(f"{nom} : meilleure acc. VC = {etude.best_value:.4f}")
    print("Meilleurs hyperparamètres :", {k: round(v, 4) for k, v in etude.best_params.items()})
    return etude

In [ ]:
# Optimiser le boosting direct
etude_direct = optimiser(*FIN_DIRECT)

In [ ]:
# Optimiser le boosting OVO (WV)
etude_ovo = optimiser(*FIN_OVO)

In [ ]:
# Optimiser la décomposition ordinale (espace élargi)
N_ESSAIS_ORD = 30
FIN_ORD = version_choisie("Boosting ordinal (Frank-Hall)", ordinal, " + sélection", avec_selection)
etude_ord = optimiser(*FIN_ORD, espace_fn=espace_large, n_essais=N_ESSAIS_ORD)

In [ ]:
# Optimiser la régression sur le rang (espace élargi)
FIN_REG = version_choisie("Régression sur le rang", regression_rang, " + sélection", avec_selection)
etude_reg = optimiser(*FIN_REG, espace_fn=espace_large, n_essais=N_ESSAIS_ORD)

In [ ]:
# Évolution de la meilleure accuracy trouvée en fonction du nombre d'essais
for nom, etude in [(FIN_DIRECT[0], etude_direct), (FIN_OVO[0], etude_ovo),
                   (FIN_ORD[0], etude_ord), (FIN_REG[0], etude_reg)]:
    plt.plot(np.maximum.accumulate([t.value for t in etude.trials]), label=nom)
plt.xlabel("Nombre d'essais"); plt.ylabel("Meilleure accuracy en VC (5 plis)")
plt.title("Optimisation bayésienne (TPE)"); plt.legend(); plt.show()

## A5 bis. Combinaison de modèles : moyenne des probabilités et stacking
Le cours présente le **stacking** comme une méthode d'ensemble qui combine des modèles de **types différents**.
Si les modèles ne se trompent pas sur les mêmes observations, leur combinaison peut faire mieux que chacun seul.
Deux stackings sont comparés :

* **Stacking actuel** (soumis précédemment, 84,2 % sur Kaggle) : boosting direct optimisé, boosting OVO-WV optimisé et
  meilleure logistique. On calcule aussi la **moyenne de leurs probabilités** (rien à estimer de plus ; pour OVO,
  les « probabilités » sont les scores WV divisés par 6).
* **Stacking ordinal** : boosting OVO-WV optimisé, décomposition ordinale optimisée et régression sur le rang
  optimisée (son score prédit, ramené entre 0 et 1, sert de variable au méta-modèle).

Dans les deux cas, un **méta-modèle** (logistique multinomiale) apprend à combiner les sorties des membres. Il est
estimé sur des sorties **hors-pli** (*out-of-fold*) : chaque observation reçoit les sorties de modèles qui ne l'ont
pas vue. Les membres gardent leurs hyperparamètres optimisés ; la pénalité C du méta-modèle est choisie par une
validation croisée interne à 5 plis (grille 0,001 à 10).

L'évaluation est une **validation croisée imbriquée** : dans chacun des 5 plis externes (les mêmes que pour toutes
les autres méthodes), les sorties hors-pli et le méta-modèle (y compris le choix de son C) sont calculés
**seulement avec les 4 autres plis**. Le pli de validation externe ne sert jamais à l'estimation, donc l'accuracy
obtenue est honnête.

In [ ]:
# Membres des deux stackings
P_LOGIT_MEILLEUR = max(P_LOGIT, key=lambda p: acc_moy(nom_logit(p)))   # C choisi par VC
param_opt = lambda fin: OPTIMISES[fin[0] + " optimisé"][1]
MEMBRES = [(FIN_DIRECT[0], param_opt(FIN_DIRECT)), (FIN_OVO[0], param_opt(FIN_OVO)),
           (nom_logit(P_LOGIT_MEILLEUR), P_LOGIT_MEILLEUR)]
MEMBRES_ORD = [(FIN_OVO[0], param_opt(FIN_OVO)), (FIN_ORD[0], param_opt(FIN_ORD)), (FIN_REG[0], param_opt(FIN_REG))]
P_STACK_ACTUEL = {"membres": MEMBRES, "nom": "Combinaison : stacking (méta-logistique)", "moyenne": True}
P_STACK_ORD = {"membres": MEMBRES_ORD, "nom": "Combinaison : stacking ordinal (méta-logistique)", "moyenne": False}
print("Stacking actuel :", [nom for nom, params in MEMBRES])
print("Stacking ordinal :", [nom for nom, params in MEMBRES_ORD])

def caract(membre, X_tr, y_tr, X_va):
    """Sorties d'un membre pour le méta-modèle : ses 4 probabilités (ou, pour la régression sur le rang,
    son score ramené entre 0 et 1). Le membre est entrainé sur (X_tr, y_tr) et appliqué à X_va."""
    nom, params = membre
    if "dérivées" in nom:
        X_tr, X_va = derivees(X_tr), derivees(X_va)
    if "sélection" in nom:
        X_tr, X_va = X_tr[VARS_UTILES], X_va[VARS_UTILES]
    if nom.startswith("Logistique"):
        return modele_logit(params["C"]).fit(X_tr, y_tr).predict_proba(X_va)
    if nom.startswith("Boosting OVO"):
        modeles = ajuster_ovo(X_tr, y_tr, params)
        return scores_ovo(modeles, X_va, "WV").to_numpy() / len(modeles)   # scores WV normalisés
    if nom.startswith("Boosting ordinal"):
        return probas_ordinal(X_tr, y_tr, X_va, params)
    if nom.startswith("Régression sur le rang"):
        return ((score_rang(X_tr, y_tr, X_va, params) - 1) / 3).reshape(-1, 1)
    return booster(params).fit(X_tr, y_tr).predict_proba(X_va)

In [ ]:
# Combinaisons estimées seulement avec l'échantillon d'entrainement (X_tr, y_tr)
def moyenne_probas(F, n_membres):
    """Classe prédite par la moyenne des probabilités des membres (F : n x (membres x 4 classes))."""
    return np.array(CLASSES)[F.reshape(len(F), n_membres, len(CLASSES)).mean(axis=1).argmax(axis=1)]

def combinaisons(X_tr, y_tr, X_va, params):
    membres = params["membres"]
    # 1) Sorties hors-pli des membres sur X_tr (5 plis internes)
    F_tr = None
    for tr, va in StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr):
        bloc = np.hstack([caract(m, X_tr.iloc[tr], y_tr.iloc[tr], X_tr.iloc[va]) for m in membres])
        if F_tr is None:
            F_tr = np.zeros((len(X_tr), bloc.shape[1]))
        F_tr[va] = bloc
    # 2) Méta-modèle (logistique multinomiale) estimé sur ces sorties hors-pli ;
    #    sa pénalité C est choisie par une VC interne à 5 plis (recherche en grille)
    meta = LogisticRegressionCV(Cs=[0.001, 0.01, 0.1, 1, 10], cv=5, scoring="accuracy",
                                max_iter=5000).fit(F_tr, y_tr)
    # 3) Membres réestimés sur tout X_tr, puis appliqués à X_va
    F_va = np.hstack([caract(m, X_tr, y_tr, X_va) for m in membres])
    sortie = {params["nom"]: meta.predict(F_va)}
    if params["moyenne"]:
        sortie["Combinaison : moyenne des probabilités"] = moyenne_probas(F_va, len(membres))
    return sortie

In [ ]:
# Validation croisée imbriquée des combinaisons (mêmes 5 plis externes que les autres méthodes)
for p in (P_STACK_ACTUEL, P_STACK_ORD):
    resultats.update(val_croisee(combinaisons, p))
noms = list(dict.fromkeys(nom + ("" if nom.startswith("Logistique") else " optimisé")
                          for nom, params in MEMBRES + MEMBRES_ORD))
noms += ["Combinaison : moyenne des probabilités", P_STACK_ACTUEL["nom"], P_STACK_ORD["nom"]]
display(resume(resultats).loc[noms].round(4))

## A6. Comparaison finale et confirmation sur le test interne
Chaque méthode est réentrainée sur les 4 000 observations de développement, puis évaluée **une seule fois**
sur les 1 000 observations mises de côté (`acc_test_interne`). Le modèle final est celui qui a la meilleure
accuracy de VC parmi les candidats (les quatre modèles optimisés, la meilleure logistique et les trois
combinaisons) : le test interne ne sert pas à choisir, seulement à confirmer.

In [ ]:
# Réentrainer chaque configuration sur les 4 000 obs. de développement et l'évaluer sur le test interne
configs = ([(naif, None), (foret, None)] + [(m, P0) for m in (direct, ova, ovo)]
           + [(avec_derivees(m), P0) for m in (direct, ova, ovo)] + [(logistique, p) for p in P_LOGIT]
           + [(m, P0) for m in (ordinal, regression_rang)]
           + [(avec_selection(m), P0) for m in (ovo, ordinal, regression_rang)])
pred_hold = {}
for methode, params in configs:
    pred_hold.update(methode(X_dev, y_dev, X_hold, params))
for nom, (methode, params) in OPTIMISES.items():
    pred_hold[nom] = methode(X_dev, y_dev, X_hold, params)[nom.replace(" optimisé", "")]
for p in (P_STACK_ACTUEL, P_STACK_ORD):
    pred_hold.update(combinaisons(X_dev, y_dev, X_hold, p))

# Tableau récapitulatif : accuracy en VC, erreur-type et accuracy sur le test interne
tableau_A = resume(resultats)
tableau_A["acc_test_interne"] = pd.Series({n: accuracy_score(y_hold, p) for n, p in pred_hold.items()})
display(tableau_A.sort_values("acc_VC", ascending=False).round(4))

In [ ]:
# Choisir le modèle final : meilleure accuracy en VC parmi les candidats
candidats = dict(OPTIMISES)
candidats[nom_logit(P_LOGIT_MEILLEUR)] = (logistique, P_LOGIT_MEILLEUR)
candidats["Combinaison : moyenne des probabilités"] = (combinaisons, P_STACK_ACTUEL)
candidats[P_STACK_ACTUEL["nom"]] = (combinaisons, P_STACK_ACTUEL)
candidats[P_STACK_ORD["nom"]] = (combinaisons, P_STACK_ORD)
display(tableau_A.loc[list(candidats)].sort_values("acc_VC", ascending=False).round(4))

NOM_FINAL = max(candidats, key=lambda n: tableau_A.loc[n, "acc_VC"])
print("Modèle retenu :", NOM_FINAL)

In [ ]:
# Matrice de confusion du modèle retenu sur le test interne (lignes = vraie classe, colonnes = prédite)
mc = confusion_matrix(y_hold, pred_hold[NOM_FINAL], labels=CLASSES)
display(pd.DataFrame(mc, index=[f"vrai {k}" for k in CLASSES], columns=[f"prédit {k}" for k in CLASSES]))
print("Accuracy par classe :", {k: round(mc[i, i] / mc[i].sum(), 3) for i, k in enumerate(CLASSES)})

**Importance des variables (VIMP) du modèle retenu**, calculée sur le test interne comme dans le chapitre
*Interprétation des modèles* : on permute une variable à la fois (5 permutations) et on mesure la baisse d'accuracy.
Le modèle est estimé une seule fois sur les 4 000 obs. de développement ; toutes les versions permutées du test
interne sont prédites en même temps. Une variable originale permutée change aussi les ratios qui l'utilisent :
on mesure donc son importance totale.

In [ ]:
# VIMP par permutation du modèle retenu sur le test interne
methode, params = candidats[NOM_FINAL]
_, vimp_final = vimp_permutation(methode, NOM_FINAL.replace(" optimisé", ""), params,
                                 X_dev, y_dev, X_hold, y_hold, R=5)
vimp_final.sort_values().plot.barh(figsize=(6, 5),
                                   title=f"VIMP du modèle retenu (baisse d'accuracy)\n{NOM_FINAL}")
plt.tight_layout(); plt.show()

## A7. Modèle final et fichier de soumission Kaggle
Le modèle retenu est réentrainé avec **les 5 000 observations** (mêmes hyperparamètres) puis appliqué aux
55 000 observations de `test.csv`. Une seule soumission finale : le tableau public (9 % des données)
ne doit pas servir à choisir le modèle, pour éviter le sur-ajustement.

In [ ]:
# Réentrainer le modèle retenu sur les 5 000 obs. et prédire les 55 000 obs. de test.csv
methode, P_FINAL = candidats[NOM_FINAL]
arrondi = lambda p: {k: round(v, 4) if isinstance(v, float) else v for k, v in p.items()}
print("Modèle final :", NOM_FINAL)
if "membres" in P_FINAL:              # combinaison : afficher ses membres
    for nom, params in P_FINAL["membres"]:
        print("  membre :", nom, arrondi(params))
else:
    print("Hyperparamètres :", arrondi(P_FINAL))
pred_test = methode(X, y, X_test, P_FINAL)[NOM_FINAL.replace(" optimisé", "")]

In [ ]:
# Créer le fichier de soumission Kaggle (colonnes id et y)
soumission = pd.DataFrame({"id": test["id"], "y": pred_test.astype(int)})
soumission.to_csv(DOSSIER + "soumission.csv", index=False)

print("Dimensions :", soumission.shape)
display(soumission.head())
display(soumission["y"].value_counts(normalize=True).sort_index().round(3).to_frame("proportion prédite"))